# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ranahassan9/Machine-Learning-FlyRank-AI/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

We do not start from every column in the table. We start from the columns we can know before the prediction window and then we engineer a clean feature matrix. The model should learn from observable page context and performance, not from the label or a future period.

The rule is simple: features must be known at the moment we predict; labels must come after.


In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Create the label in the same way the project defines it.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

# Honest feature set: only page-level signals that can be known before prediction.
numeric_features = [
    "word_count", "impressions_90d", "clicks_90d", "sessions_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_sessions_90d", "ai_traffic_pct"
]
cat_features = ["content_type", "main_intent", "impression_tier", "position_tier"]

# Prepare a clean feature table.
feature_df = df.copy()
for col in numeric_features:
    feature_df[f"{col}_missing"] = feature_df[col].isna().astype(int)
    feature_df[col] = feature_df[col].fillna(feature_df[col].median())

for col in cat_features:
    feature_df[col] = feature_df[col].fillna("missing")

model_frame = feature_df[numeric_features + [f"{c}_missing" for c in numeric_features] + cat_features]
X = pd.get_dummies(model_frame, columns=cat_features, drop_first=False)
y = feature_df["is_declining_label"]

print("Feature matrix shape:", X.shape)
print("Label mean (decline rate):", round(float(y.mean()), 4))
print("First 10 feature columns:")
print(list(X.columns[:10]))
print("\nMissing flags created:", [f"{c}_missing" for c in numeric_features[:5]])


Feature matrix shape: (30000, 37)
Label mean (decline rate): 0.5421
First 10 feature columns:
['word_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_sessions_90d', 'ai_traffic_pct']

Missing flags created: ['word_count_missing', 'impressions_90d_missing', 'clicks_90d_missing', 'sessions_90d_missing', 'ctr_missing']


## 2. Feature notes (meaning, missing, categorical, available-when?)

These notes matter because an honest model must explain what each feature means and whether it is available before the decision point.

- `word_count`: how much text exists on the page; more depth usually means more topic coverage.
- `impressions_90d`, `clicks_90d`, `sessions_90d`: observed demand over the recent 90-day window.
- `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`: rate-style signals; they are percentages, not percentages points, and must be read carefully.
- `avg_position`: search rank signal, but `0` means no position data, not a real rank of zero.
- `content_type`, `main_intent`, `impression_tier`, `position_tier`: categorical context that helps explain the page type and current ranking bracket.
- Missing entries are kept as explicit missingness flags, so we do not silently convert a lack of data into a false signal.
- Every one of these features is available before the label window and therefore is a legitimate pre-decision input.


In [3]:
feature_notes = pd.DataFrame([
    {
        "feature": "word_count",
        "meaning": "How long the page is; proxy for completeness and depth.",
        "missing_rule": "Fill median and add a missing flag.",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "impressions_90d",
        "meaning": "How often the page appeared in search over 90 days.",
        "missing_rule": "Fill median and add a missing flag.",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "ctr",
        "meaning": "Click-through rate; page relevancy signal.",
        "missing_rule": "Fill median and add a missing flag.",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "ai_sessions_90d",
        "meaning": "AI referral sessions over 90 days; a protection signal.",
        "missing_rule": "Fill median and add a missing flag.",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "content_type",
        "meaning": "Type of content page, like article or comparison page.",
        "missing_rule": "Fill as 'missing'.",
        "available_before_prediction": "Yes"
    },
    {
        "feature": "main_intent",
        "meaning": "Primary user intent for the page.",
        "missing_rule": "Fill as 'missing'.",
        "available_before_prediction": "Yes"
    }
])

print(feature_notes.to_string(index=False))


        feature                                                 meaning                        missing_rule available_before_prediction
     word_count How long the page is; proxy for completeness and depth. Fill median and add a missing flag.                         Yes
impressions_90d     How often the page appeared in search over 90 days. Fill median and add a missing flag.                         Yes
            ctr              Click-through rate; page relevancy signal. Fill median and add a missing flag.                         Yes
ai_sessions_90d AI referral sessions over 90 days; a protection signal. Fill median and add a missing flag.                         Yes
   content_type  Type of content page, like article or comparison page.                  Fill as 'missing'.                         Yes
    main_intent                       Primary user intent for the page.                  Fill as 'missing'.                         Yes


## 3. The leakage hunt

This is the most important check in the assignment. We are not only building features; we are attacking the feature list and looking for hidden answer keys.

The three common traps are:
- label-derived features
- future-window overlap
- product or decision flags that already encode the outcome

We test all three explicitly and print the result before trusting any score.


In [4]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

# Honest feature set and suspicious leakage columns.
honest_numeric = [
    "word_count", "impressions_90d", "clicks_90d", "sessions_90d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_sessions_90d", "ai_traffic_pct"
]
honest_cat = ["content_type", "main_intent", "impression_tier", "position_tier"]
leaky_cols = ["trend_pct", "trend_direction", "is_declining_label"]

feature_df = df.copy()
for col in honest_numeric:
    feature_df[f"{col}_missing"] = feature_df[col].isna().astype(int)
    feature_df[col] = feature_df[col].fillna(feature_df[col].median())
for col in honest_cat:
    feature_df[col] = feature_df[col].fillna("missing")

feature_df["trend_direction"] = feature_df["trend_direction"].fillna("unknown")
feature_df["trend_pct"] = feature_df["trend_pct"].fillna(0)

honest_cols = honest_numeric + [f"{c}_missing" for c in honest_numeric] + honest_cat
leaky_cols_all = honest_cols + ["trend_pct", "trend_direction"]

X_honest = pd.get_dummies(feature_df[honest_cols], columns=honest_cat, drop_first=False)
X_leaky = pd.get_dummies(feature_df[leaky_cols_all], columns=honest_cat + ["trend_direction"], drop_first=False)
y = feature_df["is_declining_label"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(feature_df, y, groups=feature_df["client_id"]))

model_honest = LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")
model_honest.fit(X_honest.iloc[train_idx], y.iloc[train_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], model_honest.predict_proba(X_honest.iloc[test_idx])[:, 1])

model_leaky = LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")
model_leaky.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_auc = roc_auc_score(y.iloc[test_idx], model_leaky.predict_proba(X_leaky.iloc[test_idx])[:, 1])

print("Label-derived overlap with honest features:", sorted(set(honest_cols) & set(leaky_cols)))
print("Honest AUC:", round(honest_auc, 4))
print("Leaky AUC:", round(leaky_auc, 4))
print("AUC jump from adding leakage:", round(leaky_auc - honest_auc, 4))
print("Interpretation: if the score jumps sharply after adding label-derived fields, the original model is learning the answer instead of the world.")


Label-derived overlap with honest features: []
Honest AUC: 0.548
Leaky AUC: 1.0
AUC jump from adding leakage: 0.452
Interpretation: if the score jumps sharply after adding label-derived fields, the original model is learning the answer instead of the world.


## 4. What I excluded and why

Below is the refusal list: these are fields we do not use because they are either the answer, a future view, or a product flag that would make the model circular.

- `trend_direction`: this is derived from the decline signal itself; it is the label logic in plain English.
- `trend_pct`: same issue; it is the numeric version of the target definition.
- `is_declining_label`: this is the target, not the feature.
- `content_id` / `client_id`: useful for grouping and validation, not for prediction.
- Any future-period field: we avoid using a column from a later time window than the prediction point.
- Product or decision flags: if a field reflects a business decision already made, it cannot be a feature because it is part of the old rule.


In [6]:
excluded = {
    "trend_direction": "Label-derived: it is the text version of the target definition.",
    "trend_pct": "Label-derived: it is the numeric drift measure that creates the target.",
    "is_declining_label": "Target column; never a feature.",
    "content_id": "Identifier only; used for grouping and validation, not prediction.",
    "client_id": "Grouping field for honest splits, not a predictive feature.",
    "future_window_columns": "Any later-window metric is excluded to keep the prediction time-aware and honest."
}

excluded_df = pd.DataFrame(
    [{"column": k, "reason": v} for k, v in excluded.items()]
)
print(excluded_df.to_string(index=False))


               column                                                                            reason
      trend_direction                   Label-derived: it is the text version of the target definition.
            trend_pct           Label-derived: it is the numeric drift measure that creates the target.
   is_declining_label                                                   Target column; never a feature.
           content_id                Identifier only; used for grouping and validation, not prediction.
            client_id                       Grouping field for honest splits, not a predictive feature.
future_window_columns Any later-window metric is excluded to keep the prediction time-aware and honest.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.